In [1]:
host = 'http://127.0.0.1:8000'

In [2]:
import requests


res = requests.get(f'{host}/api/rule')
rules = res.json()

rules

{'owl_test': {'schema': 'owl', 'file': 'OWL_test.txt'},
 'owl_test_1': {'schema': 'owl', 'file': 'OWL_test_1.owl'},
 'owl_test_2': {'schema': 'owl', 'file': 'OWL_test_2.owl'},
 'ids_test': {'schema': 'ids', 'file': 'IDS_StructuralSafety.txt'}}

In [3]:
rule = 'owl_test_2'

In [4]:
res = requests.get(f'{host}/api/rule/{rule}/clauses').json()

res

{'id': 'owl_test_2',
 'rationale': [{'id': 'Clause1',
   'description': 'Not an external staircase',
   'code': 'Not(ExternalStaircase)'},
  {'id': 'Clause2',
   'description': 'Not in buildings of building classes 1 and 2',
   'code': 'Not(in.some(BuildingClass1 | BuildingClass2))'},
  {'id': 'Clause3',
   'description': 'Not connecting a maximum of two storeys within the same usage unit with a total gross floor area of no more than 200 m², if a different escape route can be reached on each storey',
   'code': 'Not(connects.exactly(2, Storey & isConnectedTo.some(EscapeRoute) & in.exactly(1, UsageUnit & totalGrossFloorArea.some(ConstrainedDatatype(float, max_inclusive = 200.0)))))'}],
 'requirement': {'id': 'Clause4',
  'description': 'Must be located in a separate stairwell',
  'code': 'in.some(SeparateStairwell)'}}

In [5]:
rule = 'owl_test_1'

In [6]:
res = requests.get(f'{host}/api/rule/{rule}')

res.content

b'<?xml version="1.0"?>\n<rdf:RDF xmlns="http://test#"\n     xml:base="http://test"\n     xmlns:owl="http://www.w3.org/2002/07/owl#"\n     xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"\n     xmlns:xml="http://www.w3.org/XML/1998/namespace"\n     xmlns:xsd="http://www.w3.org/2001/XMLSchema#"\n     xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#">\n    <owl:Ontology rdf:about="http://test"/>\n    \n\n\n    <!-- \n    ///////////////////////////////////////////////////////////////////////////////////////\n    //\n    // Object Properties\n    //\n    ///////////////////////////////////////////////////////////////////////////////////////\n     -->\n\n    \n\n\n    <!-- http://test#connects -->\n\n    <owl:ObjectProperty rdf:about="http://test#connects">\n        <owl:inverseOf rdf:resource="http://test#isConnectedTo"/>\n    </owl:ObjectProperty>\n    \n\n\n    <!-- http://test#in -->\n\n    <owl:ObjectProperty rdf:about="http://test#in"/>\n    \n\n\n    <!-- http://test#isConne

In [7]:
import os

dir_path = os.path.abspath('')
rule_path = os.path.join(dir_path, 'rule.ttl')
with open(rule_path, 'wb') as file:
    file.write(res.content)

In [8]:
data_path = os.path.join(dir_path, 'test_v1.rdf')
files = {
    'data_file': open(data_path, 'rb'),
    'rule_file': open(rule_path, 'rb')
}

res = requests.post(f'{host}/api/check_model', files=files)

res.json()

{'valid': False,
 'explanation': {'necessary_staircase_0': {'storey_2': {'in': ['usage_unit_0',
     'usage_unit_1']},
   'usage_unit_1': {'totalGrossFloorArea': '209.76'},
   'necessary_staircase_0': {'Not': '(ExternalStaircase)',
    'in': 'owl:Nothing',
    'connects': ['storey_0', 'storey_1', 'storey_2', 'storey_3']},
   'storey_3': {'in': ['usage_unit_0', 'usage_unit_1']},
   'storey_1': {'in': ['usage_unit_0', 'usage_unit_1']},
   'usage_unit_0': {'totalGrossFloorArea': '209.76'}}}}